In [1]:
'''
Script permettant de classer les images dans les sous-répertoires relatifs à leur 'species', pour les champignons du Top 30 en France.

Cela permettra d'utiliser l'utilitaire 'image_dataset_from_directory' pour appeler les images dans les futurs modèles et gagner en
rapidité d'exécution.

+ AJout d'un filtre à partir de ResNet50 (ImageNet) pour retirer les images parasites

'''


"\nScript permettant de classer les images dans les sous-répertoires relatifs à leur 'species', pour les champignons du Top 30 en France.\n\nCela permettra d'utiliser l'utilitaire 'image_dataset_from_directory' pour appeler les images dans les futurs modèles et gagner en\nrapidité d'exécution.\n\n+ AJout d'un filtre à partir de ResNet50 (ImageNet) pour retirer les images parasites\n\n"

In [ ]:
# import des librairies nécessaires

import os
import shutil
import pandas as pd
import numpy as np


from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image


from sklearn.utils import resample # pour les méthodes de réchantillonnage (over-sampling et under-sampling)

In [ ]:
# Définition des chemins d'accès

rep_main_csv = 'C:\\Users\\Utilisateur\\Documents\\DataScience\\Mushroom\\' # Chemin vers le fichier csv principal (à part car trop gros pour mettre sur git)
rep_data = "C:\\Users\\Utilisateur\\Documents\\DataScience\\avr25cds_reconnaissance_champignons\\datasets\\" # Chemin vers le fichier CSV 'champignons_france_top30.csv et le dataset "final dataset_30_species.csv"
rep_img_src = 'C:\\Users\\utilisateur\\Documents\\DataScience_images_Source\\' # Chemin vers l'ensemble des images de champignons 
rep_img = 'C:\\Users\\Utilisateur\\Documents\\DataScience_images_Especes\\' # Chemin vers l'ensemble des images de champignons CLASSEES

In [ ]:
# Lecture du dataset 'observations_mushroom.csv'
mushroom = pd.read_csv(rep_main_csv+'observations_mushroom.csv', sep =  ",")

# Lecture du dataset 'top_30_France.csv'
top_30_France = pd.read_csv(rep_data+'champignons_france_top30.csv', sep = ';', encoding='latin-1')

# Fusion des 2 Datasets
mushroom = mushroom.merge(right = top_30_France, left_on='gbif_info/species', right_on='Nom scientifique', how = 'inner')

# Filtre pour un indice de confiance >= 92
mushroom = mushroom.loc[mushroom['gbif_info/confidence'] >= 92]

mushroom.to_csv(rep_data+"dataset_30_species.csv", index=False, encoding="utf-8")

# Affichage extrait du dataset et de ses caractéristiques
display(mushroom.head())
display(mushroom.info())

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_21792\1786410267.py:5: DtypeWarning: Columns (2,5,25,30) have mixed types. Specify dtype option on import or set low_memory=False.
  mushroom = pd.read_csv(rep+'observations_mushroom.csv', sep =  ",")


,image_lien,image_id,observation,label,image_url,user,date,gbif_info/kingdom,gbif_info/family,gbif_info/speciesKey,...,gbif_info/genus,gbif_info/order,thumbnail,location,gbif_info/note,gbif_info,Nom scientifique,Nom commun,Statut,Habitat typique
0,66.jpg,66,55,Trametes versicolor,http://mushroomobserver.org/images/320/66,1,2006-05-21 07:18:39,Fungi,Polyporaceae,2548311.0,...,Trametes,Polyporales,1,69.0,NaN,NaN,Trametes versicolor,Tramète versicolore,Non comestible (médicinal),Bois morts
1,67.jpg,67,56,Trametes versicolor,http://mushroomobserver.org/images/320/67,1,2006-05-21 07:18:40,Fungi,Polyporaceae,2548311.0,...,Trametes,Polyporales,0,60.0,NaN,NaN,Trametes versicolor,Tramète versicolore,Non comestible (médicinal),Bois morts
2,68.jpg,68,56,Trametes versicolor,http://mushroomobserver.org/images/320/68,1,2006-05-21 07:18:40,Fungi,Polyporaceae,2548311.0,...,Trametes,Polyporales,1,60.0,NaN,NaN,Trametes versicolor,Tramète versicolore,Non comestible (médicinal),Bois morts
3,100.jpg,100,86,Schizophyllum commune,http://mushroomobserver.org/images/320/100,1,2006-05-21 07:19:29,Fungi,Schizophyllaceae,5241128.0,...,Schizophyllum,Agaricales,1,58.0,NaN,NaN,Schizophyllum commune,Schizophylle commun,Non comestible,Bois morts
4,107.jpg,107,93,Russula virescens,http://mushroomobserver.org/images/320/107,1,2006-05-21 07:19:38,Fungi,Russulaceae,2551423.0,...,Russula,Russulales,1,53.0,NaN,NaN,Russula virescens,Russule verdoyante,Comestible,Forêts claires de feuillus


<class 'pandas.core.frame.DataFrame'>
Index: 20592 entries, 0 to 20591
Data columns (total 37 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   image_lien                20592 non-null  object 
 1   image_id                  20592 non-null  int64  
 2   observation               20592 non-null  object 
 3   label                     20592 non-null  object 
 4   image_url                 20592 non-null  object 
 5   user                      20592 non-null  object 
 6   date                      20592 non-null  object 
 7   gbif_info/kingdom         20592 non-null  object 
 8   gbif_info/family          20592 non-null  object 
 9   gbif_info/speciesKey      20592 non-null  float64
 10  gbif_info/rank            20592 non-null  object 
 11  gbif_info/phylum          20592 non-null  object 
 12  gbif_info/orderKey        20592 non-null  float64
 13  gbif_info/species         20592 non-null  object 
 14  gbif_info/c

None

In [4]:
# Simplification du fichier
mushroom = mushroom[['image_lien', 'gbif_info/species', 'Nom commun', 'Statut', 'Habitat typique']]
mushroom = mushroom.dropna()
mushroom = mushroom.rename({'image_lien' : 'image', 'gbif_info/species' : 'target'}, axis = 1)

# Affichage extrait du dataset simplifié et de ses caractéristiques
display(mushroom.head())
display(mushroom.info())

,image,target,Nom commun,Statut,Habitat typique
0,66.jpg,Trametes versicolor,Tramète versicolore,Non comestible (médicinal),Bois morts
1,67.jpg,Trametes versicolor,Tramète versicolore,Non comestible (médicinal),Bois morts
2,68.jpg,Trametes versicolor,Tramète versicolore,Non comestible (médicinal),Bois morts
3,100.jpg,Schizophyllum commune,Schizophylle commun,Non comestible,Bois morts
4,107.jpg,Russula virescens,Russule verdoyante,Comestible,Forêts claires de feuillus


<class 'pandas.core.frame.DataFrame'>
Index: 20592 entries, 0 to 20591
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   image            20592 non-null  object
 1   target           20592 non-null  object
 2   Nom commun       20592 non-null  object
 3   Statut           20592 non-null  object
 4   Habitat typique  20592 non-null  object
dtypes: object(5)
memory usage: 965.2+ KB


None

In [ ]:
# Définition des catégories qui nous intéressent
categories = mushroom['target'].unique()



In [6]:
# Nombre d'images par espèces avant over/under-sampling :

print("Nombre d'images par espèce avant over/under-sampling :\n")
for category in categories:
    order_list = mushroom.loc[mushroom['target']==category]
    count = len(order_list)
    print(f"L'espèce {category} compte {count} enregistrements")



Nombre d'images par espèce avant over/under-sampling :

L'espèce Trametes versicolor compte 1711 enregistrements
L'espèce Schizophyllum commune compte 925 enregistrements
L'espèce Russula virescens compte 169 enregistrements
L'espèce Russula sanguinea compte 234 enregistrements
L'espèce Russula cyanoxantha compte 265 enregistrements
L'espèce Pleurotus ostreatus compte 1713 enregistrements
L'espèce Marasmius oreades compte 485 enregistrements
L'espèce Macrolepiota procera compte 302 enregistrements
L'espèce Ganoderma applanatum compte 1353 enregistrements
L'espèce Craterellus lutescens compte 473 enregistrements
L'espèce Craterellus cornucopioides compte 227 enregistrements
L'espèce Coprinus comatus compte 1153 enregistrements
L'espèce Clitocybe nebularis compte 302 enregistrements
L'espèce Boletus edulis compte 2056 enregistrements
L'espèce Armillaria mellea compte 794 enregistrements
L'espèce Amanita phalloides compte 1032 enregistrements
L'espèce Amanita muscaria compte 3581 enregist

PRE-TRAITEMENT IMAGE_NET POUR RETIRER LES IMAGES PARASITES POTENTIELLES

In [8]:
# Charger le modèle ResNet50 pré-entraîné sur ImageNet
model = ResNet50(weights="imagenet")

# Liste des classes "champignons" dans ImageNet
fungi_classes = {
    "bolete", "agaric", "gyromitra", "stinkhorn",
    "earthstar", "hen-of-the-woods", "coral fungus", "mushroom"
}

def champi(img_path, top=3): # on check si il y a champignon dans le top 3 des recos
    """
    Vérifie si une image contient un champignon
    Args:
        img_path: chemin vers l'image
        top: nombre de prédictions à analyser
    Returns:
        True si l'image est identifiée comme champignon
    """
    # Charger et prétraiter l'image
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)  # permet d'avoir x.shape = (1, 224, 224, 3) compatible avec le modèle
    x = preprocess_input(x)

    # Prédictions
    preds = model.predict(x)
    decoded = decode_predictions(preds, top=top)[0]

    # Vérifie si une des prédictions est un champignon
    return any(label in fungi_classes for _, label, _ in decoded)





In [ ]:
for i, row in mushroom.iterrows():
    image_path = os.path.join(rep_img_src, row['image'])

    if champi(image_path):
        mushroom.loc[i, ["Champi"]] = True
    else :
        mushroom.loc[i, ["Champi"]] = False


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 713ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━

In [10]:
display(mushroom.head())


,image,target,Nom commun,Statut,Habitat typique,Champi
0,66.jpg,Trametes versicolor,Tramète versicolore,Non comestible (médicinal),Bois morts,True
1,67.jpg,Trametes versicolor,Tramète versicolore,Non comestible (médicinal),Bois morts,True
2,68.jpg,Trametes versicolor,Tramète versicolore,Non comestible (médicinal),Bois morts,True
3,100.jpg,Schizophyllum commune,Schizophylle commun,Non comestible,Bois morts,False
4,107.jpg,Russula virescens,Russule verdoyante,Comestible,Forêts claires de feuillus,True


In [11]:
# Filtre : on retire les images parasites
mushroom = mushroom.loc[mushroom['Champi'] == True]

In [12]:
display(mushroom.info())

<class 'pandas.core.frame.DataFrame'>
Index: 18489 entries, 0 to 20591
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   image            18489 non-null  object
 1   target           18489 non-null  object
 2   Nom commun       18489 non-null  object
 3   Statut           18489 non-null  object
 4   Habitat typique  18489 non-null  object
 5   Champi           18489 non-null  object
dtypes: object(6)
memory usage: 1011.1+ KB


None

In [13]:
# Nombre d'images par espèces après pré-filtre :

print("Nombre d'images par espèce avant over/under-sampling :\n")
for category in categories:
    order_list = mushroom.loc[mushroom['target']==category]
    count = len(order_list)
    print(f"L'espèce {category} compte {count} enregistrements")

Nombre d'images par espèce avant over/under-sampling :

L'espèce Trametes versicolor compte 1503 enregistrements
L'espèce Schizophyllum commune compte 772 enregistrements
L'espèce Russula virescens compte 150 enregistrements
L'espèce Russula sanguinea compte 210 enregistrements
L'espèce Russula cyanoxantha compte 251 enregistrements
L'espèce Pleurotus ostreatus compte 1587 enregistrements
L'espèce Marasmius oreades compte 387 enregistrements
L'espèce Macrolepiota procera compte 266 enregistrements
L'espèce Ganoderma applanatum compte 1056 enregistrements
L'espèce Craterellus lutescens compte 426 enregistrements
L'espèce Craterellus cornucopioides compte 186 enregistrements
L'espèce Coprinus comatus compte 988 enregistrements
L'espèce Clitocybe nebularis compte 284 enregistrements
L'espèce Boletus edulis compte 1945 enregistrements
L'espèce Armillaria mellea compte 739 enregistrements
L'espèce Amanita phalloides compte 953 enregistrements
L'espèce Amanita muscaria compte 3363 enregistre

In [14]:
# Paramétrage de l'under-sampling / under-sampling

# Objectif : réduire à 1000 images les catégories sur-représentées et augmenter à 500 images les catégories sous-représentées


# random_state :
rs = 42

# Liste pour stocker les DataFrames équilibrés
list_temp = []

for category in categories:
    df_category = mushroom[mushroom['target']==category].copy()
    size = len(df_category)
    
    # Under-sampling
    if size > 1000:
        df_temp = resample(
            df_category,
            replace = False,
            n_samples = 1000,
            random_state = rs
        )
    
    # Over-sampling
    elif size < 500:
        df_temp = resample(
            df_category,
            replace = True,
            n_samples = 500,
            random_state = rs
        ).reset_index(drop=True)


    # RAS si 500 <= size <= 1000
    else :
        df_temp = df_category
       

    list_temp.append(df_temp)


# Fusion de tous les DataFrames
mushroom_2 = pd.concat(list_temp, ignore_index=True)


# Vérification
print(mushroom_2['target'].value_counts())
print("Taille totale :", len(mushroom_2))

target
Trametes versicolor           1000
Pleurotus ostreatus           1000
Boletus edulis                1000
Ganoderma applanatum          1000
Amanita muscaria              1000
Coprinus comatus               988
Amanita phalloides             953
Schizophyllum commune          772
Armillaria mellea              739
Amanita rubescens              701
Agaricus campestris            563
Russula cyanoxantha            500
Russula virescens              500
Russula sanguinea              500
Clitocybe nebularis            500
Craterellus cornucopioides     500
Macrolepiota procera           500
Craterellus lutescens          500
Marasmius oreades              500
Agaricus bisporus              500
Russula vesca                  500
Russula olivacea               500
Hydnum repandum                500
Lactarius deliciosus           500
Scleroderma citrinum           500
Phallus impudicus              500
Auricularia auricula-judae     500
Russula rosea                  500
Cantharellus 

In [15]:
print(mushroom_2['image'].value_counts())

image
578119.jpg    18
481620.jpg    17
392243.jpg    17
656523.jpg    17
599455.jpg    16
              ..
146228.jpg     1
146229.jpg     1
146344.jpg     1
146348.jpg     1
118682.jpg     1
Name: count, Length: 13127, dtype: int64


In [16]:
# Création de la colonne image sans-doublon "image_unique" avec incrémentation, qui permettra de dupliquer les photos

dico = {}
new = []
for val in mushroom_2['image']:
    if val not in dico:
        dico[val] = 0
        new.append(val)
    else:
        dico[val] += 1
        new.append(f"{val.replace('.jpg', '')}_{dico[val]}.jpg")


mushroom_2['image_unique'] = new

display(mushroom_2.tail())

,image,target,Nom commun,Statut,Habitat typique,Champi,image_unique
19211,663357.jpg,Russula emetica,Russule émétique,Toxique,Forêts humides de conifères,True,663357_11.jpg
19212,459784.jpg,Russula emetica,Russule émétique,Toxique,Forêts humides de conifères,True,459784_11.jpg
19213,622199.jpg,Russula emetica,Russule émétique,Toxique,Forêts humides de conifères,True,622199_10.jpg
19214,653270.jpg,Russula emetica,Russule émétique,Toxique,Forêts humides de conifères,True,653270_8.jpg
19215,57059.jpg,Russula emetica,Russule émétique,Toxique,Forêts humides de conifères,True,57059_11.jpg


In [17]:
# Vérification
print(mushroom_2['image_unique'].describe())

count            19216
unique           19216
top       57059_11.jpg
freq                 1
Name: image_unique, dtype: object


COPIE DES IMAGES

In [ ]:
# Création des sous-dossiers pour chaque catégorie

for category in categories:
    category_path = os.path.join(rep_img, category)
    os.makedirs(category_path, exist_ok=True)

In [ ]:
# Déplacement des fichiers dans leur dossier de catégorie

for _, row in mushroom_2.iterrows():
    src_path = os.path.join(rep_img_src, row['image'])
    dst_path = os.path.join(rep_img, row['target'], row['image_unique'])

    # Vérification que le fichier existe avant de déplacer
    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)
    else:
        print(f"Fichier introuvable : {src_path}")
    

In [ ]:
# Nombre final d’images par catégories

print("Nombre d'images par espèce après over/under-sampling :\n")
for category in categories:
    final_count = len(os.listdir(rep_img+category))
    print(f"Catégorie {category} équilibrée à {final_count} images")

Nombre d'images par espèce après over/under-sampling :

Catégorie Trametes versicolor équilibrée à 1000 images
Catégorie Schizophyllum commune équilibrée à 772 images
Catégorie Russula virescens équilibrée à 500 images
Catégorie Russula sanguinea équilibrée à 500 images
Catégorie Russula cyanoxantha équilibrée à 500 images
Catégorie Pleurotus ostreatus équilibrée à 1000 images
Catégorie Marasmius oreades équilibrée à 500 images
Catégorie Macrolepiota procera équilibrée à 500 images
Catégorie Ganoderma applanatum équilibrée à 1000 images
Catégorie Craterellus lutescens équilibrée à 500 images
Catégorie Craterellus cornucopioides équilibrée à 500 images
Catégorie Coprinus comatus équilibrée à 988 images
Catégorie Clitocybe nebularis équilibrée à 500 images
Catégorie Boletus edulis équilibrée à 1000 images
Catégorie Armillaria mellea équilibrée à 739 images
Catégorie Amanita phalloides équilibrée à 953 images
Catégorie Amanita muscaria équilibrée à 1000 images
Catégorie Agaricus campestri

In [21]:
categories

array(['Trametes versicolor', 'Schizophyllum commune',
       'Russula virescens', 'Russula sanguinea', 'Russula cyanoxantha',
       'Pleurotus ostreatus', 'Marasmius oreades', 'Macrolepiota procera',
       'Ganoderma applanatum', 'Craterellus lutescens',
       'Craterellus cornucopioides', 'Coprinus comatus',
       'Clitocybe nebularis', 'Boletus edulis', 'Armillaria mellea',
       'Amanita phalloides', 'Amanita muscaria', 'Agaricus campestris',
       'Agaricus bisporus', 'Russula olivacea', 'Russula vesca',
       'Hydnum repandum', 'Lactarius deliciosus', 'Scleroderma citrinum',
       'Phallus impudicus', 'Amanita rubescens',
       'Auricularia auricula-judae', 'Russula rosea',
       'Cantharellus cibarius', 'Russula emetica'], dtype=object)